## FixBikeMVP

Redo by @anastassiavybornova

In [ ]:
# import packages
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import yaml
from itertools import combinations
import random
import numpy as np
import matplotlib.pyplot as plt
import os

### User Input:

In [ ]:
city_name = 'Copenhagen Municipality' # used for Overpass API query
proj_crs = "EPSG:25832"

radius = 2000 # in meters; used to cut-off paths for betweenness centrality compute
maxgap = 50 # in meters; maximum distance between node pairs to be considered as potential gap
penalty = {
    0: 5,
    1: 1
} # factor by which length is multiplied to get weight for NONprotected edges

### Fetch data from OSM

In [ ]:
# fetch street network data from osmnx
g = ox.graph_from_place(
    city_name, network_type='all', simplify=False
)
g = ox.simplify_graph(
    g,
    edge_attrs_differ=['highway']
)

In [ ]:
# export osmnx data to gdfs
nodes_gdf, edges_gdf = ox.graph_to_gdfs(
    g,
    nodes=True,
    edges=True,
    node_geometry=True,
    fill_edge_geometry=True
)

# project to proj_crs
nodes_gdf = nodes_gdf.to_crs(proj_crs)
edges_gdf = edges_gdf.to_crs(proj_crs)

# edges_gdf.plot()

### Map highway tags to binary classification: protected bikeinfrastructure Yes/No

currently this is done with the simple binary mapping as listed in the yaml file: `config/config_osm.yml`

In [ ]:
# first step: map all highway attributes
protected_bike_infra = yaml.load(
    open("./config/config_osm.yml"), 
    Loader=yaml.FullLoader)["protected_bike_infra"]
protected_bike_infra

In [ ]:
# add binary edge attribute "pbi" (protected bike infra: True/False)
for edge in g.edges(keys=True):
    g.edges[edge]["pbi"] = protected_bike_infra[
        g.edges[edge]["highway"]
    ]

### Drop parallel edges

g is at this stage a MultiDiGraph. We will now:
* go through all instances of parallel edges
* if they have differing `pbi` values: keep only pbi=1 edge(s) (with bikeinfra)
* if they have equal `pbi` values: *for now*, keep both & let nx take care of it through last step, which is >>
* MultiGraph to Graph conversion (keeps only 1 of each parallel edges, arbitrarily) 

In [ ]:
# to find parallel edges, get all u,v tuples for which u,v,w>0 exists:

uvs = [edge[:2] for edge in list(g.edges) if edge[2]>0] # >0 includes key=1, key=2, ...
uvs = list(set(uvs))
edges_to_drop = []

for uv in uvs:
    # collect all parallel edges for u-v node pair;
    # account for the fact that edges are directed! uv[::-1]==vu might also be on the list
    parallel_edges = [edge for edge in list(g.edges) if (edge[:2]==uv) or (edge[:2]==uv[::-1])]
    
    # get set of PBIs for this u-v parallel edge list
    pbis = set([g.edges[e]["pbi"] for e in parallel_edges])
    
    # if we have both pbi==0 and pbi==1,
    if len(pbis) == 2:

        # add edges with pbi==0 to edges_to_drop list
        to_drop = [e for e in parallel_edges if g.edges[e]["pbi"]==0]
        edges_to_drop += to_drop

edges_to_drop = list(set(edges_to_drop)) # unique list of to-drop edges

In [ ]:
# sanity check
print("edges before:", len(g.edges))
print("dropping edges:", len(edges_to_drop))
g.remove_edges_from(edges_to_drop)
print("edges after:", len(g.edges))

### Convert to undirected simple (not multi) graph

Attention: going from MultiDiGraph to Graph does 2 things:
* drop all parallel edges *randomly*
* remove all directionality (uv==vu)

FR: make this more sensitive (as to *which* of the parallel edges to drop)

In [ ]:
# Capital-G: the Graph() object we will be working with from now on
G = nx.Graph(g)

### Add weight attribute to all edges

weight = length times unprotected penalty

In [ ]:
for edge in G.edges:
    # compute edge weight
    edge_pbi = G.edges[edge]["pbi"]
    edge_length = G.edges[edge]["length"]
    edge_weight = edge_length * penalty[edge_pbi]
    # add as attribute
    G.edges[edge]["weight"] = edge_weight

### Create list of all contact nodes

Contact nodes are nodes whose edge set has a pbi set == {0,1} (i.e. that has both protected and unprotected edges)

In [ ]:
contact_nodes = []
for node in G.nodes:
    pbis = set([G.edges[edge]["pbi"] for edge in G.edges(node)])
    if len(pbis)==2:
        contact_nodes.append(node)

In [ ]:
print("contact nodes found:", len(contact_nodes))

### Create list of all potential gaps

Here defined as: contact node pair combinations **within `maxgap` euclidean distance from each other** 

In [ ]:
potential_gaps = []

for node in contact_nodes:
    node_buffer = nodes_gdf.loc[node, "geometry"].buffer(maxgap)
    q = nodes_gdf.sindex.query(node_buffer, predicate="intersects")
    neighbours = list(nodes_gdf.iloc[q].index)
    # convention: sort by ascending OSMID...
    node_pairs = [tuple(sorted(z)) for z in zip([node]*len(neighbours), neighbours)]
    potential_gaps += node_pairs

# ... so that we can easily deduplicate
potential_gaps = list(set(potential_gaps))

In [ ]:
print("potential gaps found:", len(potential_gaps))

### Determine which of the potential gaps are actual gaps by:

1. Running naive shortest path (by length) for all contact nodes
2. Creating an edge list from nodes in shortest paths (convention: `tuple(sorted([u,v]))`)
3. Only saving those gaps that have no bicycle infrastructure edges at all

In [ ]:
pbi_dict = nx.get_edge_attributes(G, "pbi")

found_gaps = []
found_gaps_nsp = [] # naive shortest paths (by length, in node list format)

for i, gap in enumerate(potential_gaps):
    u, v = gap
    nodelist = nx.shortest_path(
        G=G,
        source=u,
        target=v, 
        weight="length"
    )
    pbis = set([pbi_dict[tuple(sorted(z))] for z in zip(nodelist, nodelist[1:])])
    
    # confirm that it is an actual gap if it consists only of pbi==0 infra:
    if pbis==set([0]): 
        found_gaps.append(gap)
        found_gaps_nsp.append(nodelist)
    
    # # (manual timer)
    # if i % 100000 == 0:
    #     print(i)


### From list of gaps' nodelists, get all G edges that form part of gaps

In [ ]:
edgelist = []
for nodelist in found_gaps_nsp:
    edgelist += [tuple(sorted(z)) for z in zip(nodelist, nodelist[1:])]
# deduplicate
edgelist = list(set(edgelist))

### Weighted shortest paths for entire network (with lambda) to compute local edge betweenness centralities

In [ ]:
# set current ebc value of all G edges to 0
for edge in G.edges:
    G.edges[edge]["ebc"] = 0

# create dict that will be updated at each step
ebc = nx.get_edge_attributes(G, "ebc")

# for each node, compute "local" ebc (buffered with radius!)
# for comp feas, now only subset of randomly drawn 100 nodes
random.seed(1312)
random_nodes = random.choices(list(G.nodes), k = 100)
for node in random_nodes:
    node_buffer = nodes_gdf.loc[node, "geometry"].buffer(radius)
    q = nodes_gdf.sindex.query(node_buffer, predicate="intersects")
    neighbours = list(nodes_gdf.iloc[q].index)
    local_ebc = nx.edge_betweenness_centrality_subset(
        G=G,
        sources=[node],
        targets=neighbours,
        normalized=False, # important! otherwise the addition makes no sense
        weight="weight"# using penalty for non-pbi
    )

    # update ebc dictionary
    for k, v in local_ebc.items():
        ebc[k] += v # updating ebc!!

### Rank gaps by B

In [ ]:
Bs = []
for nodelist in found_gaps_nsp:
    edgelist = [tuple(sorted(z)) for z in zip(nodelist, nodelist[1:])]
    lengths = np.array([G.edges[edge]["length"] for edge in edgelist])
    ebcs = np.array([ebc[edge] for edge in edgelist])
    B = sum(lengths * ebcs)/sum(lengths)
    Bs.append(B)

### Order gaps

In [ ]:
df = pd.DataFrame(
    {
        "gap": found_gaps,
        "B": Bs,
        "nodelist": found_gaps_nsp
    }
)
df = df.sort_values(by="B", ascending=False).reset_index(drop=True)
df.head()

In [ ]:
# Sanity check: gap node pairs we're currently finding
fig, ax = plt.subplots(1,1, figsize = (20,20))
nodes_gdf.plot(ax=ax, markersize=0.1, color = "grey")
nodes_gdf.loc[list(set([node for gap in list(df[df.B>7000]["gap"]) for node in gap]))].plot(
    ax=ax,
    markersize=5,
    color="red"
)
ax.set_axis_off()

### Visualize gaps: will be done in separate NB, below saving the data needed for viz

In [ ]:
os.makedirs("./data/", exist_ok=True)
nodes_gdf.to_file("./data/nodes_gdf.gpkg", index=False)
edges_gdf.to_file("./data/edges_gdf.gpkg", index=False)
df.to_json("./data/df.json")